# Verify audio pipeline

Confirm an extracted `.wav` file is actually 16kHz mono, and that it
flows correctly through the frozen `wav2vec2-large-960h-lv60-self` audio
encoder.

**What "looks right" means:**
- `soundfile.info(...)` reports `samplerate == 16000` and `channels == 1`.
- The wav2vec2 processor/model runs without error on it.
- The output feature tensor has shape `(batch=1, time, hidden=1024)` --
  1024 is `wav2vec2-large`'s hidden size; `time` will be roughly
  `num_audio_samples / 320` (wav2vec2's ~20ms/frame stride at 16kHz), not
  an exact fixed number.
- The output dtype is `float32`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import soundfile as sf
import torch
from transformers import Wav2Vec2Model, Wav2Vec2Processor

/ROIHU_TYKKY_5326GQe/miniforge/envs/env1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

Point this at one real extracted `.wav` file -- e.g. one of the
`audio_path` values from notebook 01's manifests.

In [2]:
import random, pandas as pd

# Save manifests
lrs3_trainval_manifest = pd.read_csv("lrs3_trainval_manifest.csv")
lrs3_test_manifest = pd.read_csv("lrs3_test_manifest.csv")
grid_manifest = pd.read_csv("grid_manifest.csv")

# Uniformly sample one row from each manifest
lrs3_sample = lrs3_trainval_manifest.iloc[int(random.uniform(0, len(lrs3_trainval_manifest)))]
lrs3_test_manifest = lrs3_test_manifest.iloc[int(random.uniform(0, len(lrs3_test_manifest)))]
grid_sample = grid_manifest.iloc[int(random.uniform(0, len(grid_manifest)))]

# Get the corresponding audio paths
lrs3_audio_path = Path(lrs3_sample["audio_path"])
lrs3_test_manifest = Path(lrs3_test_manifest["audio_path"])
grid_audio_path = Path(grid_sample["audio_path"])

print("LRS3 trainval:", lrs3_audio_path)
print("LRS3 test:", lrs3_test_manifest)
print("GRID:", grid_audio_path)


LRS3 trainval: /scratch/project_2020712/datasets/extracted_audio/H2rG4Dg6xyI_50064.wav
LRS3 test: /scratch/project_2020712/datasets/extracted_audio/H9ZOpQzjukY_00001.wav
GRID: /scratch/project_2020712/datasets/extracted_audio/s20_srif4a.wav


## Confirm the wav is 16kHz mono

In [22]:
for aud_path in [lrs3_audio_path, lrs3_test_manifest, grid_audio_path]:
    info = sf.info(str(aud_path))
    print(f"samplerate: {info.samplerate}")
    print(f"channels:   {info.channels}")
    print(f"duration:   {info.frames / info.samplerate:.2f}s")
    
    assert info.samplerate == 16000, f"expected 16000Hz, got {info.samplerate}"
    assert info.channels == 1, f"expected mono, got {info.channels} channels"
    print("-"*10)

samplerate: 16000
channels:   1
duration:   4.42s
----------
samplerate: 16000
channels:   1
duration:   4.16s
----------
samplerate: 16000
channels:   1
duration:   2.98s
----------


## Run it through the frozen wav2vec2 encoder

In [23]:
for aud_path in [lrs3_audio_path, lrs3_test_manifest, grid_audio_path]:
    processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
    model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large-960h-lv60-self")
    model.eval()
    
    audio, sample_rate = sf.read(str(aud_path))
    inputs = processor(audio, sampling_rate=sample_rate, return_tensors="pt")
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    features = outputs.last_hidden_state
    print(f"feature tensor shape: {tuple(features.shape)}")
    print(f"feature tensor dtype: {features.dtype}")
    print("-"*10)

Loading weights: 100%|██████████| 421/421 [00:00<00:00, 48081.74it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


feature tensor shape: (1, 220, 1024)
feature tensor dtype: torch.float32
----------


Loading weights: 100%|██████████| 421/421 [00:00<00:00, 45271.17it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


feature tensor shape: (1, 207, 1024)
feature tensor dtype: torch.float32
----------


Loading weights: 100%|██████████| 421/421 [00:00<00:00, 45187.76it/s]
[transformers] Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-large-960h-lv60-self
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


feature tensor shape: (1, 148, 1024)
feature tensor dtype: torch.float32
----------


## TODO checklist

- [ ] `features.shape == (1, time, 1024)`, with `time` roughly
      `len(audio) / 320`.
- [ ] `features.dtype == torch.float32`.
- [ ] No NaN/Inf values in `features` (`torch.isfinite(features).all()`).